In [ ]:
#Tsne eeg branch analysis

In [4]:
#Tsfnet cross session
#version4
#preprocess corrected with session_id
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
# FIX (Bug 1 - Root Cause of "No valid preprocessed data found"):
# Console output said "Scanning ... on Drive D" but the path was pointing to Drive E.
# This mismatch means os.path.exists(sub_folder) returns False for every subject,
# so ALL subjects are skipped silently and total_windows stays 0 -> ValueError.
# ACTION: Update this path to match wherever your preprocessed .npy files actually live.
# Common locations to check:
#   r"D:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive D)
#   r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive E)
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter_session_id in code at the same time"  # FIX: was E:\, change to match your actual drive

ALL_SUBJECTS = list(range(1, 27))

# CROSS-SESSION CONFIGURATION (مطابق مقاله TSFNet Sec. 3.6):
# به جای تقسیم subjects به train/test (cross-subject)، برای هر subject
# یک session به عنوان test و دو session به عنوان train استفاده می‌شود.
# Hold-Out: 3 بار اجرا — هر بار یک session مختلف به عنوان test.
# نتیجه نهایی = میانگین test accuracy سه run.
# این رویکرد دقیقاً مطابق Sec. 3.6 مقاله است که cross-session را ارزیابی می‌کند.
# منبع: VPxxx_session_ids.npy که توسط generate_session_ids.py تولید شده،
# برای هر window مقدار 1، 2 یا 3 (session number) را نگه می‌دارد.

RANDOM_SEED = 42  # fixed seed for reproducible model init and window-level val split
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset",
                 session_filter=None,        # CROSS-SESSION: None=همه sessions، [1,2]=فقط session 1 و 2
                 min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.session_filter = session_filter  # CROSS-SESSION: کدام session‌ها بارگذاری شوند
        self.valid_files = []
        self.total_windows = 0
        
        filter_str = f"sessions {session_filter}" if session_filter is not None else "all sessions"
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects ({filter_str})...")
        
        # FIX (Bug 1 diagnostic guard): If PROCESSED_DATA_PATH itself does not exist,
        # every sub_folder check silently fails (os.path.exists returns False) and
        # total_windows stays 0, which raises the cryptic "No valid preprocessed data found"
        # error. This guard catches that case immediately with a clear, actionable message.
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)

                    # CROSS-SESSION: فیلتر بر اساس session_ids.npy
                    session_mask = None
                    if session_filter is not None:
                        session_ids_file = os.path.join(sub_folder, f"{subject}_session_ids.npy")
                        if os.path.exists(session_ids_file):
                            sess_ids = np.load(session_ids_file)
                            session_mask = np.isin(sess_ids, session_filter)
                            lbls = lbls[session_mask]
                        else:
                            print(f"  WARNING: {subject}_session_ids.npy not found — using all windows")

                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'session_mask': session_mask,  # CROSS-SESSION: None یا boolean mask
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        # FIX: dataset_name[:3] produced "S1_" for "S1_Train_Set" which is an invalid
        # Windows filename (trailing underscore causes OSError 22).
        # Use a sanitized slug: lowercase, replace spaces/non-alphanumeric with underscore.
        import re
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            # CROSS-SESSION: اگر session_mask وجود دارد، فقط windows مربوطه را load کن
            eeg_full  = np.load(f_info['eeg_path'])
            nirs_full = np.load(f_info['nirs_path'])
            if f_info['session_mask'] is not None:
                eeg_full  = eeg_full[f_info['session_mask']]
                nirs_full = nirs_full[f_info['session_mask']]
            self.X_eeg[s:e]  = eeg_full
            self.X_nirs[s:e] = nirs_full
            self.y[s:e]      = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    # نسخه ارتقایافته: Transformer block کامل با FFN
    # مقاله TSFNet: CAFE تنها ~2.77M پارامتر دارد (Table 6: 3.33M کل - 0.57M بدون CAFE = 2.77M).
    # نسخه قبلی فقط MultiheadAttention(64,nhead=4) بود که خیلی سبک‌تر از مقاله است.
    # این نسخه:
    #   1. d_model را به 256 افزایش می‌دهد (از 64) — اصلی‌ترین عامل افزایش پارامتر
    #   2. FFN با dim_feedforward=512 اضافه می‌کند — مثل Transformer استاندارد
    #   3. nhead=8 به جای 4 — مقاله "multi-head attention" را تأکید می‌کند
    # پارامترهای هر CrossAttention block با d_model=256, ffn=512:
    #   MHA: 256*256*3 + 256*256 ≈ 262K
    #   FFN: 256*512 + 512*256 ≈ 262K
    #   LayerNorm × 2: ~1K
    #   مجموع هر block: ~525K — دو block (cafe_attn + cafe_attn2) ≈ 1.05M
    #   با projection layers (nirs_seq_proj, gate layers): ≈ 1.3-1.5M برای کل CAFE
    def __init__(self, d_model=256, nhead=8, dropout=0.5, dim_feedforward=512):
        super().__init__()
        self.attn     = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        # FFN — مثل Transformer encoder block استاندارد
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
    def forward(self, query, key, value):
        # Cross-attention با residual
        attn_out, _ = self.attn(query, key, value)
        x = self.norm1(query + self.dropout1(attn_out))
        # FFN با residual
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x
class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # ── Fusion Branch (مقاله Table 1، ستون سوم) ──────────────────────────
        # مقاله یک شاخه Conv3d مستقل دارد که EEG tensor را می‌گیرد اما وزن‌هایش
        # کاملاً جداگانه از EEG branch یاد می‌گیرند. هدف این شاخه آماده‌سازی
        # نمایش EEG برای ترکیب با fNIRS است، نه classification خالص EEG.
        # پارامترهای kernel/stride دقیقاً مثل EEG branch (Table 1):
        #   Conv1: in (1,600,16,16) -> out (16,100,8,8)
        #   Conv2: in (16,100,8,8)  -> out (32,50,4,4)
        # خروجی پس از GAP روی محور زمانی: F_fusion ∈ R^{32×4×4}
        self.fus_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.fus_bn1   = nn.BatchNorm3d(16)
        self.fus_drop1 = nn.Dropout(dropout)
        self.fus_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.fus_bn2   = nn.BatchNorm3d(32)
        self.fus_drop2 = nn.Dropout(dropout)
        # ──────────────────────────────────────────────────────────────────────

        # ── EFGF دومرحله‌ای — اصلاح‌شده مطابق مقاله Eq.2 ───────────────────────
        # مقاله: attention map از روی feature VOLUME کامل (با بعد زمانی) ساخته می‌شود:
        #   Φ = α·σ(GAP(Conv3d(F_fnirs))) + (1-α)·σ(GAP(Conv3d(F_eeg)))
        # Conv3d روی volume کامل اعمال می‌شود، سپس GAP بعد زمانی را حذف می‌کند.
        # نتیجه: attention map فضایی (h,w) که اطلاعات زمانی را در خود جذب کرده.
        #
        # EFGF1 — روی feature volume پس از Conv1: (batch,16,100,8,8)
        # Table 1: kernel (2,2,6), stride (1,1,1) → خروجی (batch,1,8,8,~95)، سپس GAP زمانی
        self.efgf1_eeg_conv3d  = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf1_nirs_conv3d = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha1 = nn.Parameter(torch.tensor(0.0))
        self.gamma1 = nn.Parameter(torch.tensor(0.0))
        #
        # EFGF2 — روی feature volume پس از Conv2: (batch,32,50,4,4)
        # Table 1: kernel (2,2,3), stride (1,1,1) → خروجی (batch,1,4,4,~48)، سپس GAP زمانی
        self.efgf2_eeg_conv3d  = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf2_nirs_conv3d = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha2 = nn.Parameter(torch.tensor(0.0))
        self.gamma2 = nn.Parameter(torch.tensor(0.0))
        #
        # projection نهایی F_fusion (spatial 32×4×4) به 64-d برای classifier
        # CAFE روی spatial map کار می‌کند، پس flatten فقط برای fc_fusion است
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)
        # ──────────────────────────────────────────────────────────────────────

        # CAFE
        # CHANGED (Run H, 11-segment fNIRS): the CAFE temporal sequence is now the 11 REAL fNIRS
        # segments (matching the paper's actual design intent -- the whole reason for introducing
        # 11 segments was to give CAFE a genuine per-segment sequence to attend over), not a
        # 100-sample continuous-window proxy. max_len updated to match.
        # CAFE — d_model=128
        # CHANGED: از 256 به 128 کاهش یافت.
        # با 26 subject و cross-subject scenario، 1.5M پارامتر CAFE باعث overfit شدید شد
        # (train acc به 99% رسید در حالی که val acc در 54% ماند).
        # 128-d حد وسط مناسبی است: بیشتر از 64-d اولیه (capacity بیشتر)، کمتر از 256-d (overfit کمتر).
        # پارامتر CAFE با d_model=128: ~350K — در مقابل ~1.39M برای 256-d.
        CAFE_DIM = 128
        self.pos_encoder      = PositionalEncoding(CAFE_DIM, max_len=11)
        self.nirs_seq_proj    = nn.Linear(32, CAFE_DIM)
        self.fusion_cafe_proj = nn.Linear(64, CAFE_DIM)
        self.nirs_cafe_proj   = nn.Linear(64, CAFE_DIM)
        self.cafe_attn  = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate  = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_attn2 = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate2 = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_out_proj  = nn.Linear(CAFE_DIM, 64)
        self.cafe_out_proj2 = nn.Linear(CAFE_DIM, 64)

        # classifiers — two dense layers (مقاله Section 3.4: "two dense layers followed by softmax")
        # Glorot et al. (2011) JMLR: hidden layer بین feature (64-d) و output (3 کلاس)
        # از direct mapping بهتر عمل می‌کند.
        self.fc_eeg = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fnirs = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fusion = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

        # FIX (مقاله Eq.4 — Paper-Compliant Fix 1/3: Weighted combination):
        # مقاله Eq.4: "w1, w2, w3 are trainable weight coefficients obtained via a sigmoid
        # function applied to initialized parameters".
        # قبلاً: F.softmax(stack(w1,w2,w3)) که مجموع وزن‌ها را اجباراً 1 می‌کند و یک constraint
        # اضافه ایجاد می‌کند که در مقاله وجود ندارد.
        # اکنون: torch.sigmoid(wi) برای هر وزن به صورت مستقل، دقیقاً مطابق Eq.4.
        # sigmoid ∈ (0,1) بنابراین هر وزن بین 0 و 1 است اما مجموع آن‌ها ممکن است ≠ 1 باشد.
        # مقاله: "Mean denotes the averaging operation" → Mean(w1*y_eeg, w2*y_fusion, w3*y_fnirs)
        # یعنی میانگین ساده (نه weighted sum normalized) از سه پیش‌بینی وزن‌دار.
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # ── EEG branch ────────────────────────────────────────────────────────
        eeg_in = eeg.permute(0, 3, 1, 2).unsqueeze(1)       # (batch,1,600,16,16)
        ex1 = F.elu(self.eeg_bn1(self.eeg_conv1(eeg_in)))   # (batch,16,100,8,8)
        ex1 = self.eeg_drop1(ex1)
        ex2 = F.elu(self.eeg_bn2(self.eeg_conv2(ex1)))      # (batch,32,50,4,4)
        ex2 = self.eeg_drop2(ex2)
        # GAP زمانی فقط برای classifier EEG (نه برای EFGF)
        eeg_feat_map = ex2.mean(dim=2)                       # (batch,32,4,4) — برای L_EFGF و fc_eeg
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))      # (batch,64)

        # ── Fusion branch — Conv1 (قبل از EFGF1) ─────────────────────────────
        fx1 = F.elu(self.fus_bn1(self.fus_conv1(eeg_in)))   # (batch,16,100,8,8)
        fx1 = self.fus_drop1(fx1)
        # NOTE: fx2 هنوز محاسبه نمی‌شود — باید بعد از EFGF1 (fx1_updated) محاسبه شود
        # مقاله Figure 4: F̂_fusion (خروجی EFGF1) وارد Conv2 شاخه fusion می‌شود، نه fx1 اصلی

        # ── fNIRS branch ──────────────────────────────────────────────────────
        n_seg  = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)
        ny1 = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # (batch*11,16,15,8,8)
        ny1 = self.nirs_drop1(ny1)
        ny2 = F.elu(self.nirs_bn2(self.nirs_conv2(ny1)))     # (batch*11,32,8,4,4)
        ny2 = self.nirs_drop2(ny2)
        ny1 = ny1.reshape(batch_size, n_seg, 16, 15, 8, 8)
        ny2 = ny2.reshape(batch_size, n_seg, 32,  8, 4, 4)
        # GAP روی segment و زمان برای fNIRS feature map
        nirs_feat_map = ny2.mean(dim=(1, 3))                 # (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))   # (batch,64)
        y_spatial = ny2.mean(dim=(3, 4, 5))                  # (batch,11,32) — برای CAFE

        # ── EFGF1: روی feature VOLUME کامل پس از Conv1 ───────────────────────
        # مقاله Eq.2: Φ = α·σ(GAP(Conv3d(F_fnirs))) + (1-α)·σ(GAP(Conv3d(F_eeg)))
        # fNIRS: fold/unfold برای اعمال مستقل روی هر segment
        ny1_vol = ny1.mean(dim=1)                            # (batch,16,15,8,8) — mean روی segments
        # Conv3d روی volume کامل (بعد زمانی حفظ شده) → GAP زمانی → attention map فضایی
        eeg_a1_vol  = self.efgf1_eeg_conv3d(ex1)            # (batch,1,T',H',W') — T' کوچک‌شده
        nirs_a1_vol = self.efgf1_nirs_conv3d(ny1_vol)        # (batch,1,T'',H',W')
        eeg_a1  = torch.sigmoid(eeg_a1_vol.mean(dim=2))     # (batch,1,H',W') — GAP زمانی
        nirs_a1 = torch.sigmoid(nirs_a1_vol.mean(dim=2))    # (batch,1,H',W')
        a1   = torch.clamp(self.alpha1, 0, 1)
        Phi1 = a1 * nirs_a1 + (1 - a1) * eeg_a1            # (batch,1,H',W')
        # Eq.1: Fhat_fus1 = γ1·F^eeg_1 + (1-γ1)·F_fus1 + Phi1⊗F_fus1
        g1 = torch.clamp(self.gamma1, 0, 1)
        Phi1_bc = Phi1.unsqueeze(2)                          # (batch,1,1,H',W')
        fx1_updated = g1 * ex1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + (1 - g1) * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + Phi1_bc * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]]
        # (batch,16,T,H',W') — F̂_fusion خروجی EFGF1

        # ── Fusion branch — Conv2 (FIX Paper-Compliant Fix 2/3: EFGF chain) ──
        # قبلاً: fx2 = fus_conv2(fx1)  ← EFGF1 هیچ اثری روی شاخه fusion نداشت
        # اکنون: fx2 = fus_conv2(fx1_updated)  ← مطابق مقاله Figure 4
        # مقاله Figure 4: F̂_fusion (خروجی EFGF1) وارد Conv2 شاخه fusion می‌شود.
        # این تضمین می‌کند که EFGF1 واقعاً در زنجیره پردازش fusion branch اثر دارد
        # و gradient از Conv2 به EFGF1 جاری می‌شود — در نسخه قبلی این ارتباط قطع بود.
        # FIX: fx1_updated ممکن است spatial dims کوچکتر از fx1 داشته باشد (به دلیل
        # padding truncation در EFGF1). adaptive_avg_pool3d برای align کردن استفاده می‌شود
        # تا fus_conv2 با in_channels=16 و kernel صحیح کار کند.
        if fx1_updated.shape != fx1.shape:
            fx1_updated_aligned = F.adaptive_avg_pool3d(
                fx1_updated, (fx1.shape[2], fx1.shape[3], fx1.shape[4])
            )  # (batch,16,100,8,8) — همان shape fx1 اصلی
        else:
            fx1_updated_aligned = fx1_updated
        fx2 = F.elu(self.fus_bn2(self.fus_conv2(fx1_updated_aligned)))  # FIX: fx1 → fx1_updated_aligned
        fx2 = self.fus_drop2(fx2)
        # (batch,32,50,4,4) — حجم کامل برای EFGF2 حفظ می‌شود

        # ── EFGF2: روی feature VOLUME کامل پس از Conv2 ───────────────────────
        ny2_vol = ny2.mean(dim=1)                            # (batch,32,8,4,4)
        eeg_a2_vol  = self.efgf2_eeg_conv3d(ex2)            # (batch,1,T',H',W')
        nirs_a2_vol = self.efgf2_nirs_conv3d(ny2_vol)        # (batch,1,T'',H',W')
        eeg_a2  = torch.sigmoid(eeg_a2_vol.mean(dim=2))     # (batch,1,H',W')
        nirs_a2 = torch.sigmoid(nirs_a2_vol.mean(dim=2))    # (batch,1,H',W')
        a2   = torch.clamp(self.alpha2, 0, 1)
        Phi2 = a2 * nirs_a2 + (1 - a2) * eeg_a2            # (batch,1,H',W') — attention map
        g2   = torch.clamp(self.gamma2, 0, 1)
        Phi2_bc = Phi2.unsqueeze(2)                          # (batch,1,1,H',W')
        sh, sw = Phi2.shape[-2], Phi2.shape[-1]
        fx2_updated = g2 * ex2[:, :, :, :sh, :sw] \
                    + (1 - g2) * fx2[:, :, :, :sh, :sw] \
                    + Phi2_bc * fx2[:, :, :, :sh, :sw]      # (batch,32,T,H',W')
        # GAP زمانی پس از EFGF2 → spatial feature map برای CAFE و classifier
        fus_feat_map = fx2_updated.mean(dim=2)               # (batch,32,H',W')
        # H',W' ممکن است بعد از Conv3d EFGF کوچک‌تر از 4×4 شده باشند
        # adaptive pool برای تطابق با fusion_proj که انتظار 32×4×4 دارد
        fus_feat_map = F.adaptive_avg_pool2d(fus_feat_map, (4, 4))  # (batch,32,4,4)
        fused_feat   = self.fusion_proj(fus_feat_map.flatten(1))  # (batch,64)

        # برای L_EFGF (Eq.12): PCC بین eeg_feat_map و Phi2 (هم‌اندازه‌شده)
        Phi2_bc_loss = Phi2.expand(-1, 32, -1, -1)          # (batch,32,H',W')
        # اگر spatial dims با eeg_feat_map فرق دارد، به هم align می‌کنیم
        if Phi2_bc_loss.shape != eeg_feat_map.shape:
            Phi2_bc_loss = F.adaptive_avg_pool2d(Phi2_bc_loss, eeg_feat_map.shape[-2:])

        # ── CAFE Block — در فضای 128-d ──────────────────────────────────────
        nirs_seq_256  = self.nirs_seq_proj(y_spatial)            # (batch,11,128)
        nirs_proj_256 = self.pos_encoder(nirs_seq_256)           # (batch,11,128)
        fused_256     = self.fusion_cafe_proj(fused_feat)        # (batch,128)
        nirs_feat_256 = self.nirs_cafe_proj(nirs_feat)           # (batch,128)
        # Cross-Attn 2: fusion ← fNIRS
        fused_seq_256  = fused_256.unsqueeze(1)                  # (batch,1,128)
        enhanced_256   = self.cafe_attn(fused_seq_256, nirs_proj_256, nirs_proj_256).squeeze(1)
        gate2_256      = torch.sigmoid(self.cafe_gate(torch.cat([fused_256, enhanced_256], dim=-1)))
        final_feat_256 = gate2_256 * enhanced_256 + (1 - gate2_256) * fused_256
        # Cross-Attn 1: fNIRS ← fusion
        nirs_q_256     = nirs_feat_256.unsqueeze(1)
        fused_kv_256   = fused_256.unsqueeze(1)
        enh_nirs_256   = self.cafe_attn2(nirs_q_256, fused_kv_256, fused_kv_256).squeeze(1)
        gate3_256      = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat_256, enh_nirs_256], dim=-1)))
        final_nirs_256 = gate3_256 * enh_nirs_256 + (1 - gate3_256) * nirs_feat_256
        # بازگشت به 64-d برای classifier
        final_feat = self.cafe_out_proj(final_feat_256)          # (batch,64)
        final_nirs = self.cafe_out_proj2(final_nirs_256)         # (batch,64)

        # ── Classifier ────────────────────────────────────────────────────────
        eeg_out    = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out  = self.fc_fnirs(self.dropout(final_nirs))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        # FIX (مقاله Eq.4 — Paper-Compliant Fix 1/3: Weighted combination):
        # قبلاً: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        #         final_out = w[0]*eeg_out + w[1]*fnirs_out + w[2]*fusion_out
        # softmax مجموع وزن‌ها را اجباراً 1 می‌کند — این در مقاله وجود ندارد.
        # مقاله Eq.4: "w1, w2, w3 obtained via a sigmoid function"
        #             "Mean denotes the averaging operation"
        # یعنی: هر وزن مستقلاً از sigmoid می‌گذرد، سپس میانگین وزن‌دار گرفته می‌شود.
        # اکنون: sigmoid برای هر وزن، Mean برای ترکیب — دقیقاً Eq.4.
        w1 = torch.sigmoid(self.w1)  # sigmoid مطابق Eq.4
        w2 = torch.sigmoid(self.w2)
        w3 = torch.sigmoid(self.w3)
        # FIX (3-class adaptation): حذف /3 — تقسیم بر 3 logit‌ها را scale می‌کرد و
        # gradient را 3 برابر کوچک می‌کرد (gradient از w*logit → gradient/3)، که
        # یادگیری را کند و ناپایدار می‌کرد. روی argmax بی‌اثر بود اما روی loss
        # مصنوعاً مقدار را بالا نگه می‌داشت (آزمون: loss بدون /3 = 0.817 ، با /3 = 0.933).
        # weighted sum مستقیم هم از نظر ریاضی صحیح‌تر است: CrossEntropyLoss روی
        # logit کار می‌کند و نیازی به normalization قبلی ندارد.
        final_out = w1 * eeg_out + w2 * fusion_out + w3 * fnirs_out  # FIX: /3 حذف شد

        return (final_out, eeg_out, fnirs_out, fusion_out,
                eeg_feat_map, Phi2_bc_loss,
                final_nirs,   # Fe^fnirs — برای L_CAFE: nirs_feat_map در training loop
                final_feat)   # Fe^fusion — برای L_CAFE: شکل (batch,64) مثل final_nirs



# ══════════════════════════════════════════════════════════════════════════
# t-SNE DIAGNOSTIC — آیا EEG branch اصلاً feature قابل تفکیک تولید می‌کند؟
# ══════════════════════════════════════════════════════════════════════════
#
# هدف: جواب دادن به دو سوال جدا (نه یک سوال کلی):
#
#   سوال ۱ — آیا conv layer های EEG branch (قبل از bottleneck 64-بعدی eeg_fc)
#             اصلاً کلاس‌ها را از هم جدا می‌کنند؟
#             → از eeg_feat_map (512-بعدی، خروجی خام Conv2 بعد GAP زمانی) استفاده می‌شود.
#
#   سوال ۲ — آیا bottleneck 64-بعدی (Linear(512,64) که چیزی است که مقاله مشخص
#             نکرده و ما اضافه کردیم) اطلاعات discriminative را از بین می‌برد؟
#             → از eeg_feat (64-بعدی، همان چیزی که واقعاً وارد classifier می‌شود)
#             استفاده می‌شود و با سوال ۱ مقایسه می‌شود.
#
#   به‌علاوه fNIRS (final_nirs, 64-بعدی) و fusion (final_feat, 64-بعدی) هم برای
#   مقایسه‌ی نسبی (آیا واقعاً EEG از بقیه ضعیف‌تر است یا نه) رسم می‌شوند.
#
# این اسکریپت هیچ training انجام نمی‌دهد — فقط checkpoint موجود را load و
# forward pass می‌زند. باید در چند دقیقه تمام شود.

import matplotlib
matplotlib.use('Agg')  # برای اجرای بدون نمایشگر (headless/server) - اگر در Jupyter/Colab
                        # هستی و می‌خواهی inline ببینی، این خط را کامنت کن
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

# ── تنظیمات قابل تغییر ───────────────────────────────────────────────────
TEST_SESSION   = 1          # کدام session را به‌عنوان test استفاده کنیم (چک‌پوینت باید برای همین session باشد)
CKPT_PATH      = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_session{TEST_SESSION}.pth')
MAX_POINTS     = 3000       # سقف تعداد نمونه برای t-SNE (برای سرعت) — نمونه‌گیری تصادفی اگر بیشتر بود
TSNE_PERPLEXITY = 30
RANDOM_STATE   = 42
OUTPUT_FIG     = os.path.join(PROCESSED_DATA_PATH, f'tsne_eeg_branch_session{TEST_SESSION}.png')
# ──────────────────────────────────────────────────────────────────────────

def run_tsne_diagnostic():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    if not os.path.exists(CKPT_PATH):
        raise FileNotFoundError(
            f"چک‌پوینت پیدا نشد: {CKPT_PATH}\n"
            f"مطمئن شو TEST_SESSION درست تنظیم شده و آموزش برای این session قبلاً انجام شده."
        )

    # ── دیتاست تست همان session (بدون هیچ train ای) ────────────────────────
    # FIX (Windows mmap lock): نام dataset شامل pid است تا اگر اجرای قبلی به‌خاطر
    # خطا (مثلاً savefig) وسط راه کرش کرده و mmap قدیمی هنوز قفل است، اسم فایل
    # جدید با اسم قبلی تداخل/collision نداشته باشد و OSError 22 نگیریم.
    unique_tag = f"TSNE_Test_{TEST_SESSION}_{os.getpid()}"
    print(f"\nLoading test data for session {TEST_SESSION} ...")
    test_dataset = MMapHybridDataset(
        ALL_SUBJECTS, unique_tag,
        session_filter=[TEST_SESSION],
        min_windows_threshold=100
    )
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    # ── مدل + وزن‌های trained ────────────────────────────────────────────
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)
    model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
    model.eval()
    print(f"Checkpoint loaded from: {CKPT_PATH}")

    try:
        return _run_forward_and_plot(model, test_loader, device)
    finally:
        # FIX: همیشه mmap فایل‌ها را پاک/آزاد کن، حتی اگر رسم نمودار خطا بدهد،
        # تا اجرای بعدی به قفل فایل Windows برنخورد.
        test_dataset.cleanup()
        gc.collect()


def _run_forward_and_plot(model, test_loader, device):
    # ── forward pass و جمع‌آوری فیچرها ───────────────────────────────────
    eeg_512_list, eeg_64_list, nirs_64_list, fusion_64_list, labels_list, pred_list = [], [], [], [], [], []

    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b = eeg_b.to(device), nirs_b.to(device)
            (pred, eeg_out, fnirs_out, fusion_out,
             eeg_feat_map, Phi2_bc_loss, final_nirs, final_feat) = model(eeg_b, nirs_b)

            # سوال ۱: فیچر خام ۵۱۲-بعدی EEG (قبل از bottleneck)
            eeg_512 = eeg_feat_map.flatten(1)                    # (batch, 512)
            # سوال ۲: همان فیچر بعد از عبور از eeg_fc (bottleneck 64-بعدی -- همان چیزی که classifier می‌بیند)
            eeg_64  = model.eeg_fc(eeg_512)                      # (batch, 64)

            eeg_512_list.append(eeg_512.cpu().numpy())
            eeg_64_list.append(eeg_64.cpu().numpy())
            nirs_64_list.append(final_nirs.cpu().numpy())        # آنچه fc_fnirs می‌بیند
            fusion_64_list.append(final_feat.cpu().numpy())      # آنچه fc_fusion می‌بیند
            labels_list.append(lbl_b.numpy())
            pred_list.append(pred.argmax(1).cpu().numpy())

    eeg_512  = np.concatenate(eeg_512_list, axis=0)
    eeg_64   = np.concatenate(eeg_64_list, axis=0)
    nirs_64  = np.concatenate(nirs_64_list, axis=0)
    fusion_64 = np.concatenate(fusion_64_list, axis=0)
    labels   = np.concatenate(labels_list, axis=0)
    preds    = np.concatenate(pred_list, axis=0)

    print(f"\nTotal test windows: {len(labels)} | class counts: {np.bincount(labels)}")

    # ── نمونه‌گیری برای سرعت t-SNE (اگر لازم بود) ───────────────────────
    n_total = len(labels)
    if n_total > MAX_POINTS:
        rng = np.random.RandomState(RANDOM_STATE)
        idx = rng.choice(n_total, MAX_POINTS, replace=False)
        eeg_512, eeg_64, nirs_64, fusion_64 = eeg_512[idx], eeg_64[idx], nirs_64[idx], fusion_64[idx]
        labels, preds = labels[idx], preds[idx]
        print(f"Subsampled to {MAX_POINTS} windows for t-SNE speed.")

    class_names = ['0-back', '2-back', '3-back']
    colors = ['#3B82F6', '#F59E0B', '#EF4444']

    branches = {
        'EEG (pre-bottleneck, 512d)\nconv output raw': eeg_512,
        'EEG (post-bottleneck, 64d)\nwhat classifier sees': eeg_64,
        'fNIRS (post-CAFE, 64d)': nirs_64,
        'Fusion (post-CAFE, 64d)': fusion_64,
    }

    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    silhouette_results = {}

    for ax, (name, feats) in zip(axes, branches.items()):
        print(f"\nRunning t-SNE for: {name} (shape={feats.shape}) ...")
        tsne = TSNE(n_components=2, perplexity=TSNE_PERPLEXITY, random_state=RANDOM_STATE,
                    init='pca', learning_rate='auto')
        emb = tsne.fit_transform(feats)

        sil = silhouette_score(feats, labels)  # روی فضای اصلی (نه embedding 2D) — معیار واقعی جداپذیری
        silhouette_results[name] = sil

        for c in range(3):
            mask = labels == c
            ax.scatter(emb[mask, 0], emb[mask, 1], s=6, alpha=0.6,
                       color=colors[c], label=class_names[c])
        ax.set_title(f"{name}\nsilhouette={sil:.3f}", fontsize=11)
        ax.legend(fontsize=8, markerscale=2)
        ax.set_xticks([]); ax.set_yticks([])

    plt.suptitle(f"t-SNE of branch representations — Test Session {TEST_SESSION} "
                 f"(n={len(labels)} windows, {len(ALL_SUBJECTS)} subjects)", fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_FIG, dpi=150, bbox_inches='tight')
    print(f"\nFigure saved to: {OUTPUT_FIG}")

    # ── خلاصه‌ی عددی ──────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("SILHOUETTE SCORE SUMMARY (بالاتر = کلاس‌ها بهتر از هم جدا شده‌اند)")
    print("مقیاس: -1 (کاملاً قاطی) تا +1 (کاملاً جدا) | نزدیک صفر یعنی کلاس‌ها روی هم افتاده‌اند")
    print("=" * 60)
    for name, sil in silhouette_results.items():
        print(f"  {name.splitlines()[0]:40s}: {sil:+.4f}")

    print("\nتفسیر:")
    print("  - اگر eeg_512 silhouette بالاتر از eeg_64 باشد → bottleneck (eeg_fc) دارد")
    print("    اطلاعات discriminative را از بین می‌برد → باید حذفش کنی یا بزرگترش کنی.")
    print("  - اگر eeg_512 هم silhouette پایینی داشته باشد (نزدیک صفر) → مشکل از conv")
    print("    layer های EEG یا از خود سیگنال EEG است، نه از bottleneck.")
    print("  - مقایسه با fnirs/fusion silhouette نشان می‌دهد نسبت به بقیه‌ی مدل EEG واقعاً")
    print("    عقب‌تر است یا نه.")

    return {
        'silhouette': silhouette_results,
        'labels': labels,
        'preds': preds,
    }


if __name__ == "__main__":
    results = run_tsne_diagnostic()

Device: cuda

Loading test data for session 1 ...
Scanning TSNE_Test_1_23088 - 26 subjects (sessions [1])...
VP001 verified: 342 windows
VP002 verified: 342 windows
VP003 verified: 342 windows
VP004 verified: 231 windows
VP005 verified: 342 windows
VP006 verified: 342 windows
VP007 verified: 342 windows
VP008 verified: 342 windows
VP009 verified: 342 windows
VP010 verified: 342 windows
VP011 verified: 342 windows
VP012 verified: 342 windows
VP013 verified: 339 windows
VP014 verified: 336 windows
VP015 verified: 340 windows
VP016 verified: 342 windows
VP017 verified: 342 windows
VP018 verified: 342 windows
VP019 verified: 342 windows
VP020 verified: 342 windows
VP021 verified: 336 windows
VP022 verified: 342 windows
VP023 verified: 342 windows
VP024 verified: 342 windows
VP025 verified: 342 windows
VP026 verified: 342 windows
Streaming clean data to disk memory-map (TSNE_Test_1_23088)...
--> TSNE_Test_1_23088 Ready on Disk (CLEAN)! Total Windows: 8764 | Labels: [2927 2919 2918]

Checkpo

In [ ]:
#Tsne subject vs class

In [1]:
#Tsfnet cross session
#version4
#preprocess corrected with session_id
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
# FIX (Bug 1 - Root Cause of "No valid preprocessed data found"):
# Console output said "Scanning ... on Drive D" but the path was pointing to Drive E.
# This mismatch means os.path.exists(sub_folder) returns False for every subject,
# so ALL subjects are skipped silently and total_windows stays 0 -> ValueError.
# ACTION: Update this path to match wherever your preprocessed .npy files actually live.
# Common locations to check:
#   r"D:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive D)
#   r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive E)
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter_session_id in code at the same time"  # FIX: was E:\, change to match your actual drive

ALL_SUBJECTS = list(range(1, 27))

# CROSS-SESSION CONFIGURATION (مطابق مقاله TSFNet Sec. 3.6):
# به جای تقسیم subjects به train/test (cross-subject)، برای هر subject
# یک session به عنوان test و دو session به عنوان train استفاده می‌شود.
# Hold-Out: 3 بار اجرا — هر بار یک session مختلف به عنوان test.
# نتیجه نهایی = میانگین test accuracy سه run.
# این رویکرد دقیقاً مطابق Sec. 3.6 مقاله است که cross-session را ارزیابی می‌کند.
# منبع: VPxxx_session_ids.npy که توسط generate_session_ids.py تولید شده،
# برای هر window مقدار 1، 2 یا 3 (session number) را نگه می‌دارد.

RANDOM_SEED = 42  # fixed seed for reproducible model init and window-level val split
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset",
                 session_filter=None,        # CROSS-SESSION: None=همه sessions، [1,2]=فقط session 1 و 2
                 min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.session_filter = session_filter  # CROSS-SESSION: کدام session‌ها بارگذاری شوند
        self.valid_files = []
        self.total_windows = 0
        
        filter_str = f"sessions {session_filter}" if session_filter is not None else "all sessions"
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects ({filter_str})...")
        
        # FIX (Bug 1 diagnostic guard): If PROCESSED_DATA_PATH itself does not exist,
        # every sub_folder check silently fails (os.path.exists returns False) and
        # total_windows stays 0, which raises the cryptic "No valid preprocessed data found"
        # error. This guard catches that case immediately with a clear, actionable message.
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)

                    # CROSS-SESSION: فیلتر بر اساس session_ids.npy
                    session_mask = None
                    if session_filter is not None:
                        session_ids_file = os.path.join(sub_folder, f"{subject}_session_ids.npy")
                        if os.path.exists(session_ids_file):
                            sess_ids = np.load(session_ids_file)
                            session_mask = np.isin(sess_ids, session_filter)
                            lbls = lbls[session_mask]
                        else:
                            print(f"  WARNING: {subject}_session_ids.npy not found — using all windows")

                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'session_mask': session_mask,  # CROSS-SESSION: None یا boolean mask
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        # FIX: dataset_name[:3] produced "S1_" for "S1_Train_Set" which is an invalid
        # Windows filename (trailing underscore causes OSError 22).
        # Use a sanitized slug: lowercase, replace spaces/non-alphanumeric with underscore.
        import re
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            # CROSS-SESSION: اگر session_mask وجود دارد، فقط windows مربوطه را load کن
            eeg_full  = np.load(f_info['eeg_path'])
            nirs_full = np.load(f_info['nirs_path'])
            if f_info['session_mask'] is not None:
                eeg_full  = eeg_full[f_info['session_mask']]
                nirs_full = nirs_full[f_info['session_mask']]
            self.X_eeg[s:e]  = eeg_full
            self.X_nirs[s:e] = nirs_full
            self.y[s:e]      = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    # return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)
    # CHANGED: direct inspection of the paper's PDF (glyph/font-level check of the delimiter
    # characters surrounding PCC(...) in Eq. 12 and Eq. 13 -- font CMEX10, the LaTeX extensible-
    # delimiter font used for \left| ... \right| -- confirmed via a >>1000% zoom render of both
    # equations) shows BOTH equations use |PCC(...)|, i.e. the absolute value of the correlation.
    # This means strong NEGATIVE correlation should also be rewarded (minimized loss), not
    # penalized -- only the correlation's magnitude (how strongly related the two feature maps are)
    # matters, not its sign.
    return 1.0 - torch.abs(pcc)  # CHANGED: 1.0 - pcc -> 1.0 - |pcc|, per confirmed Eq. 12/13 form


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    # نسخه ارتقایافته: Transformer block کامل با FFN
    # مقاله TSFNet: CAFE تنها ~2.77M پارامتر دارد (Table 6: 3.33M کل - 0.57M بدون CAFE = 2.77M).
    # نسخه قبلی فقط MultiheadAttention(64,nhead=4) بود که خیلی سبک‌تر از مقاله است.
    # این نسخه:
    #   1. d_model را به 256 افزایش می‌دهد (از 64) — اصلی‌ترین عامل افزایش پارامتر
    #   2. FFN با dim_feedforward=512 اضافه می‌کند — مثل Transformer استاندارد
    #   3. nhead=8 به جای 4 — مقاله "multi-head attention" را تأکید می‌کند
    # پارامترهای هر CrossAttention block با d_model=256, ffn=512:
    #   MHA: 256*256*3 + 256*256 ≈ 262K
    #   FFN: 256*512 + 512*256 ≈ 262K
    #   LayerNorm × 2: ~1K
    #   مجموع هر block: ~525K — دو block (cafe_attn + cafe_attn2) ≈ 1.05M
    #   با projection layers (nirs_seq_proj, gate layers): ≈ 1.3-1.5M برای کل CAFE
    def __init__(self, d_model=256, nhead=8, dropout=0.5, dim_feedforward=512):
        super().__init__()
        self.attn     = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        # FFN — مثل Transformer encoder block استاندارد
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
    def forward(self, query, key, value):
        # Cross-attention با residual
        attn_out, _ = self.attn(query, key, value)
        x = self.norm1(query + self.dropout1(attn_out))
        # FFN با residual
        x = self.norm2(x + self.dropout2(self.ffn(x)))
        return x
class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()

        # NOTE: eeg_ch is kept only for call-site backward compatibility (the training script still
        # instantiates TSFNet_With_fNIRS_2D(eeg_ch=28, ...)). It is UNUSED below now: EEG input is
        # the spatially-mapped (16, 16, 600) grid tensor (see forward()), a single-channel 3D volume
        # (channel=1, since it is one interpolated grid, not 28 separate channel-space signals
        # anymore), so eeg_conv1's in_channels is fixed to 1, not eeg_ch.

        # EEG branch
        self.eeg_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.eeg_bn1 = nn.BatchNorm3d(16)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.eeg_bn2 = nn.BatchNorm3d(32)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_fc = nn.Linear(32 * 4 * 4, 64)

        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(6, 4, 4), stride=(2, 2, 2), padding=(2, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 2, 2), stride=(2, 2, 2), padding=(1, 0, 0))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_fc = nn.Linear(32 * 4 * 4, 64)

        # ── Fusion Branch (مقاله Table 1، ستون سوم) ──────────────────────────
        # مقاله یک شاخه Conv3d مستقل دارد که EEG tensor را می‌گیرد اما وزن‌هایش
        # کاملاً جداگانه از EEG branch یاد می‌گیرند. هدف این شاخه آماده‌سازی
        # نمایش EEG برای ترکیب با fNIRS است، نه classification خالص EEG.
        # پارامترهای kernel/stride دقیقاً مثل EEG branch (Table 1):
        #   Conv1: in (1,600,16,16) -> out (16,100,8,8)
        #   Conv2: in (16,100,8,8)  -> out (32,50,4,4)
        # خروجی پس از GAP روی محور زمانی: F_fusion ∈ R^{32×4×4}
        self.fus_conv1 = nn.Conv3d(1, 16, kernel_size=(12, 4, 4), stride=(6, 2, 2), padding=(3, 1, 1))
        self.fus_bn1   = nn.BatchNorm3d(16)
        self.fus_drop1 = nn.Dropout(dropout)
        self.fus_conv2 = nn.Conv3d(16, 32, kernel_size=(6, 2, 2), stride=(2, 2, 2), padding=(2, 0, 0))
        self.fus_bn2   = nn.BatchNorm3d(32)
        self.fus_drop2 = nn.Dropout(dropout)
        # ──────────────────────────────────────────────────────────────────────

        # ── EFGF دومرحله‌ای — اصلاح‌شده مطابق مقاله Eq.2 ───────────────────────
        # مقاله: attention map از روی feature VOLUME کامل (با بعد زمانی) ساخته می‌شود:
        #   Φ = α·σ(GAP(Conv3d(F_fnirs))) + (1-α)·σ(GAP(Conv3d(F_eeg)))
        # Conv3d روی volume کامل اعمال می‌شود، سپس GAP بعد زمانی را حذف می‌کند.
        # نتیجه: attention map فضایی (h,w) که اطلاعات زمانی را در خود جذب کرده.
        #
        # EFGF1 — روی feature volume پس از Conv1: (batch,16,100,8,8)
        # Table 1: kernel (2,2,6), stride (1,1,1) → خروجی (batch,1,8,8,~95)، سپس GAP زمانی
        self.efgf1_eeg_conv3d  = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf1_nirs_conv3d = nn.Conv3d(16, 1, kernel_size=(6, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha1 = nn.Parameter(torch.tensor(0.0))
        self.gamma1 = nn.Parameter(torch.tensor(0.0))
        #
        # EFGF2 — روی feature volume پس از Conv2: (batch,32,50,4,4)
        # Table 1: kernel (2,2,3), stride (1,1,1) → خروجی (batch,1,4,4,~48)، سپس GAP زمانی
        self.efgf2_eeg_conv3d  = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.efgf2_nirs_conv3d = nn.Conv3d(32, 1, kernel_size=(3, 2, 2), stride=(1, 1, 1), padding=(0, 0, 0))
        self.alpha2 = nn.Parameter(torch.tensor(0.0))
        self.gamma2 = nn.Parameter(torch.tensor(0.0))
        #
        # projection نهایی F_fusion (spatial 32×4×4) به 64-d برای classifier
        # CAFE روی spatial map کار می‌کند، پس flatten فقط برای fc_fusion است
        self.fusion_proj = nn.Linear(32 * 4 * 4, 64)
        # ──────────────────────────────────────────────────────────────────────

        # CAFE
        # CHANGED (Run H, 11-segment fNIRS): the CAFE temporal sequence is now the 11 REAL fNIRS
        # segments (matching the paper's actual design intent -- the whole reason for introducing
        # 11 segments was to give CAFE a genuine per-segment sequence to attend over), not a
        # 100-sample continuous-window proxy. max_len updated to match.
        # CAFE — d_model=128
        # CHANGED: از 256 به 128 کاهش یافت.
        # با 26 subject و cross-subject scenario، 1.5M پارامتر CAFE باعث overfit شدید شد
        # (train acc به 99% رسید در حالی که val acc در 54% ماند).
        # 128-d حد وسط مناسبی است: بیشتر از 64-d اولیه (capacity بیشتر)، کمتر از 256-d (overfit کمتر).
        # پارامتر CAFE با d_model=128: ~350K — در مقابل ~1.39M برای 256-d.
        CAFE_DIM = 128
        self.pos_encoder      = PositionalEncoding(CAFE_DIM, max_len=11)
        self.nirs_seq_proj    = nn.Linear(32, CAFE_DIM)
        self.fusion_cafe_proj = nn.Linear(64, CAFE_DIM)
        self.nirs_cafe_proj   = nn.Linear(64, CAFE_DIM)
        self.cafe_attn  = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate  = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_attn2 = CrossAttention(CAFE_DIM, nhead=8, dropout=dropout, dim_feedforward=256)
        self.cafe_gate2 = nn.Linear(CAFE_DIM * 2, CAFE_DIM)
        self.cafe_out_proj  = nn.Linear(CAFE_DIM, 64)
        self.cafe_out_proj2 = nn.Linear(CAFE_DIM, 64)

        # classifiers — two dense layers (مقاله Section 3.4: "two dense layers followed by softmax")
        # Glorot et al. (2011) JMLR: hidden layer بین feature (64-d) و output (3 کلاس)
        # از direct mapping بهتر عمل می‌کند.
        self.fc_eeg = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fnirs = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
        self.fc_fusion = nn.Sequential(
            nn.Linear(64, 32), nn.ELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )

        # FIX (مقاله Eq.4 — Paper-Compliant Fix 1/3: Weighted combination):
        # مقاله Eq.4: "w1, w2, w3 are trainable weight coefficients obtained via a sigmoid
        # function applied to initialized parameters".
        # قبلاً: F.softmax(stack(w1,w2,w3)) که مجموع وزن‌ها را اجباراً 1 می‌کند و یک constraint
        # اضافه ایجاد می‌کند که در مقاله وجود ندارد.
        # اکنون: torch.sigmoid(wi) برای هر وزن به صورت مستقل، دقیقاً مطابق Eq.4.
        # sigmoid ∈ (0,1) بنابراین هر وزن بین 0 و 1 است اما مجموع آن‌ها ممکن است ≠ 1 باشد.
        # مقاله: "Mean denotes the averaging operation" → Mean(w1*y_eeg, w2*y_fusion, w3*y_fnirs)
        # یعنی میانگین ساده (نه weighted sum normalized) از سه پیش‌بینی وزن‌دار.
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)

        # ── EEG branch ────────────────────────────────────────────────────────
        eeg_in = eeg.permute(0, 3, 1, 2).unsqueeze(1)       # (batch,1,600,16,16)
        ex1 = F.elu(self.eeg_bn1(self.eeg_conv1(eeg_in)))   # (batch,16,100,8,8)
        ex1 = self.eeg_drop1(ex1)
        ex2 = F.elu(self.eeg_bn2(self.eeg_conv2(ex1)))      # (batch,32,50,4,4)
        ex2 = self.eeg_drop2(ex2)
        # GAP زمانی فقط برای classifier EEG (نه برای EFGF)
        eeg_feat_map = ex2.mean(dim=2)                       # (batch,32,4,4) — برای L_EFGF و fc_eeg
        eeg_feat = self.eeg_fc(eeg_feat_map.flatten(1))      # (batch,64)

        # ── Fusion branch — Conv1 (قبل از EFGF1) ─────────────────────────────
        fx1 = F.elu(self.fus_bn1(self.fus_conv1(eeg_in)))   # (batch,16,100,8,8)
        fx1 = self.fus_drop1(fx1)
        # NOTE: fx2 هنوز محاسبه نمی‌شود — باید بعد از EFGF1 (fx1_updated) محاسبه شود
        # مقاله Figure 4: F̂_fusion (خروجی EFGF1) وارد Conv2 شاخه fusion می‌شود، نه fx1 اصلی

        # ── fNIRS branch ──────────────────────────────────────────────────────
        n_seg  = nirs.size(1)
        nirs_r = nirs.permute(0, 1, 5, 4, 2, 3)
        nirs_r = nirs_r.reshape(batch_size * n_seg, 2, 30, 16, 16)
        ny1 = F.elu(self.nirs_bn1(self.nirs_conv1(nirs_r)))  # (batch*11,16,15,8,8)
        ny1 = self.nirs_drop1(ny1)
        ny2 = F.elu(self.nirs_bn2(self.nirs_conv2(ny1)))     # (batch*11,32,8,4,4)
        ny2 = self.nirs_drop2(ny2)
        ny1 = ny1.reshape(batch_size, n_seg, 16, 15, 8, 8)
        ny2 = ny2.reshape(batch_size, n_seg, 32,  8, 4, 4)
        # GAP روی segment و زمان برای fNIRS feature map
        nirs_feat_map = ny2.mean(dim=(1, 3))                 # (batch,32,4,4)
        nirs_feat = self.nirs_fc(nirs_feat_map.flatten(1))   # (batch,64)
        y_spatial = ny2.mean(dim=(3, 4, 5))                  # (batch,11,32) — برای CAFE

        # ── EFGF1: روی feature VOLUME کامل پس از Conv1 ───────────────────────
        # مقاله Eq.2: Φ = α·σ(GAP(Conv3d(F_fnirs))) + (1-α)·σ(GAP(Conv3d(F_eeg)))
        # fNIRS: fold/unfold برای اعمال مستقل روی هر segment
        ny1_vol = ny1.mean(dim=1)                            # (batch,16,15,8,8) — mean روی segments
        # Conv3d روی volume کامل (بعد زمانی حفظ شده) → GAP زمانی → attention map فضایی
        eeg_a1_vol  = self.efgf1_eeg_conv3d(ex1)            # (batch,1,T',H',W') — T' کوچک‌شده
        nirs_a1_vol = self.efgf1_nirs_conv3d(ny1_vol)        # (batch,1,T'',H',W')
        eeg_a1  = torch.sigmoid(eeg_a1_vol.mean(dim=2))     # (batch,1,H',W') — GAP زمانی
        nirs_a1 = torch.sigmoid(nirs_a1_vol.mean(dim=2))    # (batch,1,H',W')
        a1   = torch.clamp(self.alpha1, 0, 1)
        Phi1 = a1 * nirs_a1 + (1 - a1) * eeg_a1            # (batch,1,H',W')
        # Eq.1: Fhat_fus1 = γ1·F^eeg_1 + (1-γ1)·F_fus1 + Phi1⊗F_fus1
        g1 = torch.clamp(self.gamma1, 0, 1)
        Phi1_bc = Phi1.unsqueeze(2)                          # (batch,1,1,H',W')
        fx1_updated = g1 * ex1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + (1 - g1) * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]] \
                    + Phi1_bc * fx1[:, :, :, :Phi1.shape[-2], :Phi1.shape[-1]]
        # (batch,16,T,H',W') — F̂_fusion خروجی EFGF1

        # ── Fusion branch — Conv2 (FIX Paper-Compliant Fix 2/3: EFGF chain) ──
        # قبلاً: fx2 = fus_conv2(fx1)  ← EFGF1 هیچ اثری روی شاخه fusion نداشت
        # اکنون: fx2 = fus_conv2(fx1_updated)  ← مطابق مقاله Figure 4
        # مقاله Figure 4: F̂_fusion (خروجی EFGF1) وارد Conv2 شاخه fusion می‌شود.
        # این تضمین می‌کند که EFGF1 واقعاً در زنجیره پردازش fusion branch اثر دارد
        # و gradient از Conv2 به EFGF1 جاری می‌شود — در نسخه قبلی این ارتباط قطع بود.
        # FIX: fx1_updated ممکن است spatial dims کوچکتر از fx1 داشته باشد (به دلیل
        # padding truncation در EFGF1). adaptive_avg_pool3d برای align کردن استفاده می‌شود
        # تا fus_conv2 با in_channels=16 و kernel صحیح کار کند.
        if fx1_updated.shape != fx1.shape:
            fx1_updated_aligned = F.adaptive_avg_pool3d(
                fx1_updated, (fx1.shape[2], fx1.shape[3], fx1.shape[4])
            )  # (batch,16,100,8,8) — همان shape fx1 اصلی
        else:
            fx1_updated_aligned = fx1_updated
        fx2 = F.elu(self.fus_bn2(self.fus_conv2(fx1_updated_aligned)))  # FIX: fx1 → fx1_updated_aligned
        fx2 = self.fus_drop2(fx2)
        # (batch,32,50,4,4) — حجم کامل برای EFGF2 حفظ می‌شود

        # ── EFGF2: روی feature VOLUME کامل پس از Conv2 ───────────────────────
        ny2_vol = ny2.mean(dim=1)                            # (batch,32,8,4,4)
        eeg_a2_vol  = self.efgf2_eeg_conv3d(ex2)            # (batch,1,T',H',W')
        nirs_a2_vol = self.efgf2_nirs_conv3d(ny2_vol)        # (batch,1,T'',H',W')
        eeg_a2  = torch.sigmoid(eeg_a2_vol.mean(dim=2))     # (batch,1,H',W')
        nirs_a2 = torch.sigmoid(nirs_a2_vol.mean(dim=2))    # (batch,1,H',W')
        a2   = torch.clamp(self.alpha2, 0, 1)
        Phi2 = a2 * nirs_a2 + (1 - a2) * eeg_a2            # (batch,1,H',W') — attention map
        g2   = torch.clamp(self.gamma2, 0, 1)
        Phi2_bc = Phi2.unsqueeze(2)                          # (batch,1,1,H',W')
        sh, sw = Phi2.shape[-2], Phi2.shape[-1]
        fx2_updated = g2 * ex2[:, :, :, :sh, :sw] \
                    + (1 - g2) * fx2[:, :, :, :sh, :sw] \
                    + Phi2_bc * fx2[:, :, :, :sh, :sw]      # (batch,32,T,H',W')
        # GAP زمانی پس از EFGF2 → spatial feature map برای CAFE و classifier
        fus_feat_map = fx2_updated.mean(dim=2)               # (batch,32,H',W')
        # H',W' ممکن است بعد از Conv3d EFGF کوچک‌تر از 4×4 شده باشند
        # adaptive pool برای تطابق با fusion_proj که انتظار 32×4×4 دارد
        fus_feat_map = F.adaptive_avg_pool2d(fus_feat_map, (4, 4))  # (batch,32,4,4)
        fused_feat   = self.fusion_proj(fus_feat_map.flatten(1))  # (batch,64)

        # برای L_EFGF (Eq.12): PCC بین eeg_feat_map و Phi2 (هم‌اندازه‌شده)
        Phi2_bc_loss = Phi2.expand(-1, 32, -1, -1)          # (batch,32,H',W')
        # اگر spatial dims با eeg_feat_map فرق دارد، به هم align می‌کنیم
        if Phi2_bc_loss.shape != eeg_feat_map.shape:
            Phi2_bc_loss = F.adaptive_avg_pool2d(Phi2_bc_loss, eeg_feat_map.shape[-2:])

        # ── CAFE Block — در فضای 128-d ──────────────────────────────────────
        nirs_seq_256  = self.nirs_seq_proj(y_spatial)            # (batch,11,128)
        nirs_proj_256 = self.pos_encoder(nirs_seq_256)           # (batch,11,128)
        fused_256     = self.fusion_cafe_proj(fused_feat)        # (batch,128)
        nirs_feat_256 = self.nirs_cafe_proj(nirs_feat)           # (batch,128)
        # Cross-Attn 2: fusion ← fNIRS
        fused_seq_256  = fused_256.unsqueeze(1)                  # (batch,1,128)
        enhanced_256   = self.cafe_attn(fused_seq_256, nirs_proj_256, nirs_proj_256).squeeze(1)
        gate2_256      = torch.sigmoid(self.cafe_gate(torch.cat([fused_256, enhanced_256], dim=-1)))
        final_feat_256 = gate2_256 * enhanced_256 + (1 - gate2_256) * fused_256
        # Cross-Attn 1: fNIRS ← fusion
        nirs_q_256     = nirs_feat_256.unsqueeze(1)
        fused_kv_256   = fused_256.unsqueeze(1)
        enh_nirs_256   = self.cafe_attn2(nirs_q_256, fused_kv_256, fused_kv_256).squeeze(1)
        gate3_256      = torch.sigmoid(self.cafe_gate2(torch.cat([nirs_feat_256, enh_nirs_256], dim=-1)))
        final_nirs_256 = gate3_256 * enh_nirs_256 + (1 - gate3_256) * nirs_feat_256
        # بازگشت به 64-d برای classifier
        final_feat = self.cafe_out_proj(final_feat_256)          # (batch,64)
        final_nirs = self.cafe_out_proj2(final_nirs_256)         # (batch,64)

        # ── Classifier ────────────────────────────────────────────────────────
        eeg_out    = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out  = self.fc_fnirs(self.dropout(final_nirs))
        fusion_out = self.fc_fusion(self.dropout(final_feat))

        # FIX (مقاله Eq.4 — Paper-Compliant Fix 1/3: Weighted combination):
        # قبلاً: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        #         final_out = w[0]*eeg_out + w[1]*fnirs_out + w[2]*fusion_out
        # softmax مجموع وزن‌ها را اجباراً 1 می‌کند — این در مقاله وجود ندارد.
        # مقاله Eq.4: "w1, w2, w3 obtained via a sigmoid function"
        #             "Mean denotes the averaging operation"
        # یعنی: هر وزن مستقلاً از sigmoid می‌گذرد، سپس میانگین وزن‌دار گرفته می‌شود.
        # اکنون: sigmoid برای هر وزن، Mean برای ترکیب — دقیقاً Eq.4.
        w1 = torch.sigmoid(self.w1)  # sigmoid مطابق Eq.4
        w2 = torch.sigmoid(self.w2)
        w3 = torch.sigmoid(self.w3)
        # FIX (3-class adaptation): حذف /3 — تقسیم بر 3 logit‌ها را scale می‌کرد و
        # gradient را 3 برابر کوچک می‌کرد (gradient از w*logit → gradient/3)، که
        # یادگیری را کند و ناپایدار می‌کرد. روی argmax بی‌اثر بود اما روی loss
        # مصنوعاً مقدار را بالا نگه می‌داشت (آزمون: loss بدون /3 = 0.817 ، با /3 = 0.933).
        # weighted sum مستقیم هم از نظر ریاضی صحیح‌تر است: CrossEntropyLoss روی
        # logit کار می‌کند و نیازی به normalization قبلی ندارد.
        final_out = w1 * eeg_out + w2 * fusion_out + w3 * fnirs_out  # FIX: /3 حذف شد

        return (final_out, eeg_out, fnirs_out, fusion_out,
                eeg_feat_map, Phi2_bc_loss,
                final_nirs,   # Fe^fnirs — برای L_CAFE: nirs_feat_map در training loop
                final_feat)   # Fe^fusion — برای L_CAFE: شکل (batch,64) مثل final_nirs



# ══════════════════════════════════════════════════════════════════════════
# آزمایش دوم — Subject-ID vs Class-Label
# ══════════════════════════════════════════════════════════════════════════
#
# فرضیه‌ای که تست می‌کنیم:
#   آیا واریانس بین‌سوژه‌ای (تفاوت‌های فیزیولوژیک/سیگنال بین ۲۶ نفر) آنقدر قوی
#   است که سیگنال کلاس (۰-back/۲-back/۳-back) را در فضای فیچر EEG "خفه" می‌کند؟
#
# روش: همان embedding های t-SNE قبلی را این بار به‌جای رنگ‌بندی بر اساس کلاس،
# بر اساس subject_id رنگ می‌کنیم. اگر subject ها خیلی واضح‌تر از کلاس‌ها از هم
# جدا شدند → EEG branch دارد بیشتر «کی هست» را یاد می‌گیرد تا «چه کاری می‌کند».
#
# این اسکریپت از صفر forward pass می‌زند (چون embedding قبلی ذخیره نشده بود)،
# ولی سریع است — چند دقیقه، بدون هیچ training.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder

# ── تنظیمات ──────────────────────────────────────────────────────────────
TEST_SESSION     = 1
CKPT_PATH        = os.path.join(PROCESSED_DATA_PATH, f'best_tsfnet_session{TEST_SESSION}.pth')
MAX_POINTS       = 3000
TSNE_PERPLEXITY  = 30
RANDOM_STATE     = 42
OUTPUT_FIG       = os.path.join(PROCESSED_DATA_PATH, f'tsne_subject_vs_class_session{TEST_SESSION}.png')
# ──────────────────────────────────────────────────────────────────────────


def get_subject_ids_array(dataset):
    """
    از dataset.valid_files یک آرایه‌ی طول-کل می‌سازد که برای هر window
    (به همان ترتیبی که در X_eeg/X_nirs/y چیده شده) اسم subject را نگه می‌دارد.
    این ترتیب دقیقاً همان ترتیبی است که DataLoader با shuffle=False برمی‌گرداند،
    پس می‌توان مستقیم با labels/preds هم‌تراز کرد.
    """
    subj_ids = np.empty(dataset.total_windows, dtype=object)
    for f_info in dataset.valid_files:
        subj_ids[f_info['start_idx']:f_info['end_idx']] = f_info['subject']
    return subj_ids


def run_subject_vs_class_diagnostic():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    if not os.path.exists(CKPT_PATH):
        raise FileNotFoundError(f"چک‌پوینت پیدا نشد: {CKPT_PATH}")

    unique_tag = f"SubjVsClass_{TEST_SESSION}_{os.getpid()}"
    print(f"\nLoading test data for session {TEST_SESSION} ...")
    test_dataset = MMapHybridDataset(
        ALL_SUBJECTS, unique_tag,
        session_filter=[TEST_SESSION],
        min_windows_threshold=100
    )

    try:
        return _forward_and_plot(test_dataset, device)
    finally:
        test_dataset.cleanup()
        gc.collect()


def _forward_and_plot(test_dataset, device):
    subject_ids_full = get_subject_ids_array(test_dataset)  # طول = total_windows، هم‌تراز با dataset

    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)
    model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
    model.eval()
    print(f"Checkpoint loaded from: {CKPT_PATH}")

    eeg_512_list, eeg_64_list, labels_list = [], [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b = eeg_b.to(device), nirs_b.to(device)
            (pred, eeg_out, fnirs_out, fusion_out,
             eeg_feat_map, Phi2_bc_loss, final_nirs, final_feat) = model(eeg_b, nirs_b)

            eeg_512 = eeg_feat_map.flatten(1)
            eeg_64  = model.eeg_fc(eeg_512)

            eeg_512_list.append(eeg_512.cpu().numpy())
            eeg_64_list.append(eeg_64.cpu().numpy())
            labels_list.append(lbl_b.numpy())

    eeg_512 = np.concatenate(eeg_512_list, axis=0)
    eeg_64  = np.concatenate(eeg_64_list, axis=0)
    labels  = np.concatenate(labels_list, axis=0)

    assert len(subject_ids_full) == len(labels), \
        "طول subject_ids با طول labels یکی نیست — چک کن ترتیب dataset عوض نشده باشد."

    print(f"\nTotal test windows: {len(labels)} | subjects: {len(np.unique(subject_ids_full))}")

    # ── نمونه‌گیری یکسان برای هر دو نمایش (کلاس و subject) ────────────────
    n_total = len(labels)
    if n_total > MAX_POINTS:
        rng = np.random.RandomState(RANDOM_STATE)
        idx = rng.choice(n_total, MAX_POINTS, replace=False)
        eeg_512, eeg_64 = eeg_512[idx], eeg_64[idx]
        labels = labels[idx]
        subject_ids = subject_ids_full[idx]
        print(f"Subsampled to {MAX_POINTS} windows.")
    else:
        subject_ids = subject_ids_full

    class_names = ['0-back', '2-back', '3-back']
    class_colors = ['#3B82F6', '#F59E0B', '#EF4444']

    le = LabelEncoder()
    subject_int = le.fit_transform(subject_ids)  # 0..25
    n_subjects = len(le.classes_)
    subj_cmap = cm.get_cmap('gist_ncar', n_subjects)  # ۲۶ رنگ به‌قدر کافی متمایز

    branches = {
        'EEG pre-bottleneck (512d)': eeg_512,
        'EEG post-bottleneck (64d)': eeg_64,
    }

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    sil_summary = {}

    for col, (name, feats) in enumerate(branches.items()):
        print(f"\nRunning t-SNE for: {name} ...")
        tsne = TSNE(n_components=2, perplexity=TSNE_PERPLEXITY, random_state=RANDOM_STATE,
                    init='pca', learning_rate='auto')
        emb = tsne.fit_transform(feats)

        sil_class   = silhouette_score(feats, labels)
        sil_subject = silhouette_score(feats, subject_int)
        sil_summary[name] = {'by_class': sil_class, 'by_subject': sil_subject}

        # ردیف بالا: رنگ‌بندی بر اساس کلاس (مثل آزمایش قبلی)
        ax_top = axes[0, col]
        for c in range(3):
            mask = labels == c
            ax_top.scatter(emb[mask, 0], emb[mask, 1], s=6, alpha=0.6,
                           color=class_colors[c], label=class_names[c])
        ax_top.set_title(f"{name}\nرنگ = کلاس | silhouette={sil_class:.3f}", fontsize=11)
        ax_top.legend(fontsize=8, markerscale=2)
        ax_top.set_xticks([]); ax_top.set_yticks([])

        # ردیف پایین: رنگ‌بندی بر اساس subject
        ax_bot = axes[1, col]
        sc = ax_bot.scatter(emb[:, 0], emb[:, 1], s=6, alpha=0.7,
                            c=subject_int, cmap=subj_cmap, vmin=0, vmax=n_subjects - 1)
        ax_bot.set_title(f"{name}\nرنگ = Subject ID | silhouette={sil_subject:.3f}", fontsize=11)
        ax_bot.set_xticks([]); ax_bot.set_yticks([])
        plt.colorbar(sc, ax=ax_bot, label='subject index', fraction=0.046)

    plt.suptitle(f"Class-separation vs Subject-separation — EEG branch, Session {TEST_SESSION}\n"
                 f"(n={len(labels)} windows, {n_subjects} subjects)", fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_FIG, dpi=150, bbox_inches='tight')
    print(f"\nFigure saved to: {OUTPUT_FIG}")

    print("\n" + "=" * 65)
    print("SILHOUETTE: CLASS vs SUBJECT (کدوم قوی‌تر جدا می‌شه؟)")
    print("=" * 65)
    for name, vals in sil_summary.items():
        print(f"  {name:30s} | by_class: {vals['by_class']:+.4f}  |  by_subject: {vals['by_subject']:+.4f}")

    print("\nتفسیر:")
    print("  - اگر by_subject خیلی بزرگ‌تر از by_class باشد (مثلاً +0.3 در برابر -0.03)،")
    print("    یعنی مدل عمدتاً دارد هویت subject را یاد می‌گیرد، نه کلاس شناختی.")
    print("    → باید به سراغ subject-normalization یا adversarial de-biasing بروی.")
    print("  - اگر by_subject هم نزدیک صفر یا منفی بود، یعنی مشکل صرفاً از overwhelm")
    print("    شدن با subject نیست و باید فرضیه‌های دیگر (کیفیت سیگنال EEG برای این")
    print("    تسک خاص، طول پنجره‌ی ۳ ثانیه، عدم تفکیک باند فرکانسی) را دنبال کنی.")

    return sil_summary


if __name__ == "__main__":
    results = run_subject_vs_class_diagnostic()

Device: cuda

Loading test data for session 1 ...
Scanning SubjVsClass_1_18200 - 26 subjects (sessions [1])...
VP001 verified: 342 windows
VP002 verified: 342 windows
VP003 verified: 342 windows
VP004 verified: 231 windows
VP005 verified: 342 windows
VP006 verified: 342 windows
VP007 verified: 342 windows
VP008 verified: 342 windows
VP009 verified: 342 windows
VP010 verified: 342 windows
VP011 verified: 342 windows
VP012 verified: 342 windows
VP013 verified: 339 windows
VP014 verified: 336 windows
VP015 verified: 340 windows
VP016 verified: 342 windows
VP017 verified: 342 windows
VP018 verified: 342 windows
VP019 verified: 342 windows
VP020 verified: 342 windows
VP021 verified: 336 windows
VP022 verified: 342 windows
VP023 verified: 342 windows
VP024 verified: 342 windows
VP025 verified: 342 windows
VP026 verified: 342 windows
Streaming clean data to disk memory-map (SubjVsClass_1_18200)...
--> SubjVsClass_1_18200 Ready on Disk (CLEAN)! Total Windows: 8764 | Labels: [2927 2919 2918]

C

In [ ]:
#Bandpower quick check

In [3]:
#Tsfnet cross session
#version4
#preprocess corrected with session_id
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
# FIX (Bug 1 - Root Cause of "No valid preprocessed data found"):
# Console output said "Scanning ... on Drive D" but the path was pointing to Drive E.
# This mismatch means os.path.exists(sub_folder) returns False for every subject,
# so ALL subjects are skipped silently and total_windows stays 0 -> ValueError.
# ACTION: Update this path to match wherever your preprocessed .npy files actually live.
# Common locations to check:
#   r"D:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive D)
#   r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive E)
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter_session_id in code at the same time"  # FIX: was E:\, change to match your actual drive

ALL_SUBJECTS = list(range(1, 27))

# CROSS-SESSION CONFIGURATION (مطابق مقاله TSFNet Sec. 3.6):
# به جای تقسیم subjects به train/test (cross-subject)، برای هر subject
# یک session به عنوان test و دو session به عنوان train استفاده می‌شود.
# Hold-Out: 3 بار اجرا — هر بار یک session مختلف به عنوان test.
# نتیجه نهایی = میانگین test accuracy سه run.
# این رویکرد دقیقاً مطابق Sec. 3.6 مقاله است که cross-session را ارزیابی می‌کند.
# منبع: VPxxx_session_ids.npy که توسط generate_session_ids.py تولید شده،
# برای هر window مقدار 1، 2 یا 3 (session number) را نگه می‌دارد.

RANDOM_SEED = 42  # fixed seed for reproducible model init and window-level val split
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset",
                 session_filter=None,        # CROSS-SESSION: None=همه sessions، [1,2]=فقط session 1 و 2
                 min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.session_filter = session_filter  # CROSS-SESSION: کدام session‌ها بارگذاری شوند
        self.valid_files = []
        self.total_windows = 0
        
        filter_str = f"sessions {session_filter}" if session_filter is not None else "all sessions"
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects ({filter_str})...")
        
        # FIX (Bug 1 diagnostic guard): If PROCESSED_DATA_PATH itself does not exist,
        # every sub_folder check silently fails (os.path.exists returns False) and
        # total_windows stays 0, which raises the cryptic "No valid preprocessed data found"
        # error. This guard catches that case immediately with a clear, actionable message.
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)

                    # CROSS-SESSION: فیلتر بر اساس session_ids.npy
                    session_mask = None
                    if session_filter is not None:
                        session_ids_file = os.path.join(sub_folder, f"{subject}_session_ids.npy")
                        if os.path.exists(session_ids_file):
                            sess_ids = np.load(session_ids_file)
                            session_mask = np.isin(sess_ids, session_filter)
                            lbls = lbls[session_mask]
                        else:
                            print(f"  WARNING: {subject}_session_ids.npy not found — using all windows")

                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'session_mask': session_mask,  # CROSS-SESSION: None یا boolean mask
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        # FIX: dataset_name[:3] produced "S1_" for "S1_Train_Set" which is an invalid
        # Windows filename (trailing underscore causes OSError 22).
        # Use a sanitized slug: lowercase, replace spaces/non-alphanumeric with underscore.
        import re
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            # CROSS-SESSION: اگر session_mask وجود دارد، فقط windows مربوطه را load کن
            eeg_full  = np.load(f_info['eeg_path'])
            nirs_full = np.load(f_info['nirs_path'])
            if f_info['session_mask'] is not None:
                eeg_full  = eeg_full[f_info['session_mask']]
                nirs_full = nirs_full[f_info['session_mask']]
            self.X_eeg[s:e]  = eeg_full
            self.X_nirs[s:e] = nirs_full
            self.y[s:e]      = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# ══════════════════════════════════════════════════════════════════════════
# آزمایش سریع: آیا Band-Power (theta/alpha/beta/...) اطلاعات کلاس بیشتری از
# دامنه‌ی خام سیگنال EEG دارد؟
# ══════════════════════════════════════════════════════════════════════════
#
# روش: به‌جای TSFNet کامل (که هر بار ~۶ ساعت طول می‌کشد)، یک classifier خطی
# ساده (LogisticRegression) روی دو نوع فیچر مقایسه می‌شود:
#
#   A) Broadband power  — کل انرژی سیگنال هر پیکسل grid (بدون تفکیک فرکانسی)
#      → این معادل چیزی است که TSFNet با conv روی سیگنال خام می‌بیند (تقریبی).
#   B) Band power        — انرژی تفکیک‌شده در ۵ باند (delta/theta/alpha/beta/gamma)
#      → دقیقاً همان چیزی که MSTFDN (مقاله مقایسه‌ای گزارش تو) استفاده می‌کند.
#
# اگر (B) به‌طور معنادار از (A) بهتر عمل کند، یعنی مشکل EEG branch واقعاً از
# نبود تفکیک باند فرکانسی است — و مسیر بعدی معماری‌ات مشخص می‌شود.
#
# ارزیابی با GroupKFold بر اساس subject انجام می‌شود (نه random split) چون
# پنجره‌ها با گام ۱ ثانیه/طول ۳ ثانیه همپوشانی ۶۶٪ دارند — یک random split
# باعث نشت اطلاعات بین train/test می‌شود.
#
# ⚠️ توجه: این ارزیابی subject-independent است (کل subject در یک طرف)، در حالی
# که ارزیابی اصلی TSFNet شما subject-dependent/cross-session بود (همان subject
# در train و test، فقط session فرق دارد). پس عدد دقیق قابل‌مقایسه‌ی مستقیم با
# 45% نیست — آنچه مهم است "جهت" و "اندازه‌ی نسبی بهبود" بین A و B است، نه عدد مطلق.

import scipy.signal as sig
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ── تنظیمات ──────────────────────────────────────────────────────────────
TEST_SESSION   = 1          # کدام session بارگذاری شود (فقط برای این آزمایش دیتای خام لازم است، نه checkpoint)
FS             = 200        # فرکانس نمونه‌برداری EEG بعد از downsample (مقاله TSFNet Sec 2.1)
MAX_WINDOWS    = 6000        # سقف تعداد پنجره برای سرعت (Welch روی همه‌ی 256 پیکسل، خیلی سریع است ولی محتاط باشیم)
N_FOLDS        = 5           # تعداد subject-group برای cross-validation
RANDOM_STATE   = 42

BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 45),
}
# ──────────────────────────────────────────────────────────────────────────


def get_subject_ids_array(dataset):
    subj_ids = np.empty(dataset.total_windows, dtype=object)
    for f_info in dataset.valid_files:
        subj_ids[f_info['start_idx']:f_info['end_idx']] = f_info['subject']
    return subj_ids


def compute_band_power_features(eeg_windows, fs=FS, batch_size=300):
    """
    eeg_windows: (n_windows, 16, 16, 600) numpy array
    خروجی:
        broadband_feats: (n_windows, 256)         -- کل توان هر پیکسل grid
        band_feats:      (n_windows, 256 * n_bands) -- توان تفکیک‌شده در هر باند

    FIX (MemoryError): scipy.signal.welch وقتی کل آرایه را یک‌جا می‌گیرد، یک آرایه‌ی
    میانی STFT به شکل (n_windows, 256, n_freq, n_segments) می‌سازد که برای 6000 پنجره
    نزدیک به 6GB حافظه لازم دارد (float64). راه‌حل: پردازش دسته‌ای (batch) -- هر بار
    فقط batch_size پنجره را به Welch می‌دهیم، نتیجه را جمع می‌کنیم، و آرایه‌ی میانی
    دسته‌ی قبلی از حافظه آزاد می‌شود. این کار مقدار محاسبات را عوض نمی‌کند، فقط پیک
    مصرف حافظه را از ~6GB به چند صد مگابایت کاهش می‌دهد.
    """
    n_windows = eeg_windows.shape[0]
    x_full = eeg_windows.reshape(n_windows, 256, 600)   # (n_windows, 256, 600)

    print(f"  Running Welch PSD on {n_windows} windows x 256 virtual channels "
          f"(batched, batch_size={batch_size}) ...")

    broadband_chunks = []
    band_chunks = []
    freqs = None  # از اولین batch گرفته می‌شود، برای همه یکسان است

    for start in range(0, n_windows, batch_size):
        end = min(start + batch_size, n_windows)
        x_batch = x_full[start:end]                     # (b, 256, 600)

        f, psd = sig.welch(x_batch, fs=fs, nperseg=128, noverlap=64, axis=-1)
        if freqs is None:
            freqs = f

        broadband_mask = (freqs >= 0.5) & (freqs <= 50)
        broadband_power = psd[:, :, broadband_mask].sum(axis=-1)          # (b, 256)
        broadband_chunks.append(np.log10(broadband_power + 1e-12))

        band_feats_list = []
        for band_name, (lo, hi) in BANDS.items():
            mask = (freqs >= lo) & (freqs <= hi)
            band_power = psd[:, :, mask].sum(axis=-1)                     # (b, 256)
            band_feats_list.append(np.log10(band_power + 1e-12))
        band_chunks.append(np.concatenate(band_feats_list, axis=1))       # (b, 256*5)

        del psd, x_batch  # آزادسازی صریح آرایه‌ی میانی بزرگ این batch
        if (start // batch_size) % 5 == 0:
            print(f"    processed {end}/{n_windows} windows ...")

    broadband_feats = np.concatenate(broadband_chunks, axis=0)
    band_feats = np.concatenate(band_chunks, axis=0)

    return broadband_feats, band_feats


def run_bandpower_check():
    print(f"\nLoading raw EEG windows for session {TEST_SESSION} ...")
    unique_tag = f"BandPowerCheck_{TEST_SESSION}_{os.getpid()}"
    dataset = MMapHybridDataset(
        ALL_SUBJECTS, unique_tag,
        session_filter=[TEST_SESSION],
        min_windows_threshold=100
    )

    try:
        return _extract_and_evaluate(dataset)
    finally:
        dataset.cleanup()
        gc.collect()


def _extract_and_evaluate(dataset):
    subject_ids_full = get_subject_ids_array(dataset)
    y_full = dataset.y.copy()
    n_total = dataset.total_windows

    # ── نمونه‌گیری برای سرعت (طبقه‌بندی‌شده تصادفی، نه بر اساس subject) ────
    if n_total > MAX_WINDOWS:
        rng = np.random.RandomState(RANDOM_STATE)
        idx = rng.choice(n_total, MAX_WINDOWS, replace=False)
        idx.sort()  # حفظ ترتیب برای خواندن راحت‌تر از memmap
    else:
        idx = np.arange(n_total)

    print(f"Using {len(idx)} windows out of {n_total} total.")
    eeg_windows = np.array(dataset.X_eeg[idx])   # (n, 16, 16, 600) -- می‌آید از memmap
    y = y_full[idx]
    subject_ids = subject_ids_full[idx]

    print(f"Class counts in sample: {np.bincount(y)}")
    print(f"Subjects in sample: {len(np.unique(subject_ids))}")

    broadband_feats, band_feats = compute_band_power_features(eeg_windows)
    print(f"Broadband feature shape: {broadband_feats.shape}")
    print(f"Band-power feature shape: {band_feats.shape}")

    groups = subject_ids
    gkf = GroupKFold(n_splits=N_FOLDS)

    clf_broadband = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
    clf_bandpower = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))

    print(f"\nRunning {N_FOLDS}-fold GroupKFold (grouped by subject) cross-validation ...")
    print("(هر subject کامل یا در train یا در test — بدون نشت اطلاعات بین پنجره‌های هم‌پوشان)")

    scores_broadband = cross_val_score(clf_broadband, broadband_feats, y, groups=groups,
                                        cv=gkf, scoring='accuracy', n_jobs=-1)
    scores_bandpower = cross_val_score(clf_bandpower, band_feats, y, groups=groups,
                                        cv=gkf, scoring='accuracy', n_jobs=-1)

    print("\n" + "=" * 65)
    print("RESULTS — Linear classifier (LogisticRegression), subject-grouped CV")
    print("=" * 65)
    print(f"  Broadband power (256 feats, no band split): "
          f"{scores_broadband.mean()*100:.2f}% ± {scores_broadband.std()*100:.2f}%  "
          f"(per fold: {[f'{s*100:.1f}%' for s in scores_broadband]})")
    print(f"  Band power      (1280 feats, 5 bands):      "
          f"{scores_bandpower.mean()*100:.2f}% ± {scores_bandpower.std()*100:.2f}%  "
          f"(per fold: {[f'{s*100:.1f}%' for s in scores_bandpower]})")
    print(f"  Chance level (3 classes):                    33.33%")

    diff = (scores_bandpower.mean() - scores_broadband.mean()) * 100
    print(f"\n  Δ (band-power − broadband): {diff:+.2f} percentage points")

    print("\nتفسیر:")
    print("  - اگر Δ مثبت و قابل‌توجه باشد (مثلاً بیش از ۲-۳ واحد درصد، با توجه به std")
    print("    فولدها)، یعنی تفکیک باند فرکانسی واقعاً اطلاعات کلاس بیشتری آشکار می‌کند")
    print("    نسبت به دامنه‌ی خام/broadband -- این تأیید می‌کند مسیر بعدی معماری باید")
    print("    band-decomposition باشد (شبیه MSTFDN).")
    print("  - اگر Δ نزدیک صفر یا منفی بود، یعنی مشکل EEG صرفاً از نبود تفکیک باند")
    print("    نیست -- شاید طول پنجره (۳ ثانیه ممکن است برای تفکیک باند theta/alpha")
    print("    کافی نباشد) یا ماهیت خود سیگنال n-back در این دیتاست باشد.")
    print("  - این عدد subject-independent است (GroupKFold)، نه subject-dependent مثل")
    print("    TSFNet شما -- پس با ۴۵٪ مستقیم مقایسه نکن، فقط جهت Δ مهم است.")

    return {
        'broadband_scores': scores_broadband,
        'bandpower_scores': scores_bandpower,
        'delta_pp': diff,
    }


if __name__ == "__main__":
    results = run_bandpower_check()


Loading raw EEG windows for session 1 ...
Scanning BandPowerCheck_1_18200 - 26 subjects (sessions [1])...
VP001 verified: 342 windows
VP002 verified: 342 windows
VP003 verified: 342 windows
VP004 verified: 231 windows
VP005 verified: 342 windows
VP006 verified: 342 windows
VP007 verified: 342 windows
VP008 verified: 342 windows
VP009 verified: 342 windows
VP010 verified: 342 windows
VP011 verified: 342 windows
VP012 verified: 342 windows
VP013 verified: 339 windows
VP014 verified: 336 windows
VP015 verified: 340 windows
VP016 verified: 342 windows
VP017 verified: 342 windows
VP018 verified: 342 windows
VP019 verified: 342 windows
VP020 verified: 342 windows
VP021 verified: 336 windows
VP022 verified: 342 windows
VP023 verified: 342 windows
VP024 verified: 342 windows
VP025 verified: 342 windows
VP026 verified: 342 windows
Streaming clean data to disk memory-map (BandPowerCheck_1_18200)...
--> BandPowerCheck_1_18200 Ready on Disk (CLEAN)! Total Windows: 8764 | Labels: [2927 2919 2918]


In [ ]:
#Longer window bandpower check

In [2]:
#Tsfnet cross session
#version4
#preprocess corrected with session_id
#subject-level val + lr=0.0001 + weight_decay=0.1 + label_smoothing + two dense layers+ WEIGHT_DECAY_GRID = [0.01, 0.02, 0.05]
#Tsfnet model run h 11segments
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
from sklearn.metrics import cohen_kappa_score  # NEW: adopted from reviewing a third-party implementation -- corrects raw accuracy for chance agreement, a standard BCI-literature metric
import warnings
import json  # NEW: for results.json export, adopted from reviewing a third-party implementation
warnings.filterwarnings('ignore')

# CONFIGURATION
# FIX (Bug 1 - Root Cause of "No valid preprocessed data found"):
# Console output said "Scanning ... on Drive D" but the path was pointing to Drive E.
# This mismatch means os.path.exists(sub_folder) returns False for every subject,
# so ALL subjects are skipped silently and total_windows stays 0 -> ValueError.
# ACTION: Update this path to match wherever your preprocessed .npy files actually live.
# Common locations to check:
#   r"D:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive D)
#   r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"   (Drive E)
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\preproc_P2_no_fnirs_filter_session_id in code at the same time"  # FIX: was E:\, change to match your actual drive

ALL_SUBJECTS = list(range(1, 27))

# CROSS-SESSION CONFIGURATION (مطابق مقاله TSFNet Sec. 3.6):
# به جای تقسیم subjects به train/test (cross-subject)، برای هر subject
# یک session به عنوان test و دو session به عنوان train استفاده می‌شود.
# Hold-Out: 3 بار اجرا — هر بار یک session مختلف به عنوان test.
# نتیجه نهایی = میانگین test accuracy سه run.
# این رویکرد دقیقاً مطابق Sec. 3.6 مقاله است که cross-session را ارزیابی می‌کند.
# منبع: VPxxx_session_ids.npy که توسط generate_session_ids.py تولید شده،
# برای هر window مقدار 1، 2 یا 3 (session number) را نگه می‌دارد.

RANDOM_SEED = 42  # fixed seed for reproducible model init and window-level val split
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset",
                 session_filter=None,        # CROSS-SESSION: None=همه sessions، [1,2]=فقط session 1 و 2
                 min_windows_threshold=500):
        super().__init__()
        self.dataset_name = dataset_name
        self.session_filter = session_filter  # CROSS-SESSION: کدام session‌ها بارگذاری شوند
        self.valid_files = []
        self.total_windows = 0
        
        filter_str = f"sessions {session_filter}" if session_filter is not None else "all sessions"
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects ({filter_str})...")
        
        # FIX (Bug 1 diagnostic guard): If PROCESSED_DATA_PATH itself does not exist,
        # every sub_folder check silently fails (os.path.exists returns False) and
        # total_windows stays 0, which raises the cryptic "No valid preprocessed data found"
        # error. This guard catches that case immediately with a clear, actionable message.
        if not os.path.exists(PROCESSED_DATA_PATH):
            raise FileNotFoundError(
                f"\n[PATH ERROR] PROCESSED_DATA_PATH does not exist:\n"
                f"  '{PROCESSED_DATA_PATH}'\n"
                f"  Check your drive letter (D: vs E:) and folder structure.\n"
                f"  Expected subject folders: VP001, VP002, ... inside that directory."
            )
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)

                    # CROSS-SESSION: فیلتر بر اساس session_ids.npy
                    session_mask = None
                    if session_filter is not None:
                        session_ids_file = os.path.join(sub_folder, f"{subject}_session_ids.npy")
                        if os.path.exists(session_ids_file):
                            sess_ids = np.load(session_ids_file)
                            session_mask = np.isin(sess_ids, session_filter)
                            lbls = lbls[session_mask]
                        else:
                            print(f"  WARNING: {subject}_session_ids.npy not found — using all windows")

                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'session_mask': session_mask,  # CROSS-SESSION: None یا boolean mask
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

        # FIX: dataset_name[:3] produced "S1_" for "S1_Train_Set" which is an invalid
        # Windows filename (trailing underscore causes OSError 22).
        # Use a sanitized slug: lowercase, replace spaces/non-alphanumeric with underscore.
        import re
        safe_slug = re.sub(r'[^a-z0-9]', '_', dataset_name.lower())[:12]
        self.mmap_eeg_path  = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{safe_slug}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        # self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        # CHANGED (EEG spatial mapping): preprocessed EEG windows are now (16, 16, 600) grid tensors
        # (azimuthal-equidistant-projected onto a 16x16 spatial grid, per TSFNet paper Sec. 2.3),
        # not (28, 600) raw channel-space tensors. This memmap shape MUST match the .npy files
        # actually saved on disk (VPxxx_eeg_windows.npy, confirmed (N, 16, 16, 600) via the VP001-
        # VP003 preprocessing runs), or np.load()-ing them into this memmap slice would fail with a
        # shape/broadcast error the moment training data is loaded.
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 16, 16, 600))  # CHANGED: (N,28,600) -> (N,16,16,600)
        # self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        # CHANGED (Run H, 11-segment fNIRS): matches the new preprocessing output shape
        # (11, 16, 16, 30, 2) per window -- 11 overlapping 3s segments (current + subsequent 10,
        # per Zhang et al. 2025 Sec. 2.3: "each EEG time window was paired with the current and
        # subsequent 10 fNIRS segments... each fNIRS sample had a shape of 11 x 36 x 30 x 2"),
        # instead of the previous single continuous 10s window (2, 100, 16, 16).
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 11, 16, 16, 30, 2))  # CHANGED
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            # CROSS-SESSION: اگر session_mask وجود دارد، فقط windows مربوطه را load کن
            eeg_full  = np.load(f_info['eeg_path'])
            nirs_full = np.load(f_info['nirs_path'])
            if f_info['session_mask'] is not None:
                eeg_full  = eeg_full[f_info['session_mask']]
                nirs_full = nirs_full[f_info['session_mask']]
            self.X_eeg[s:e]  = eeg_full
            self.X_nirs[s:e] = nirs_full
            self.y[s:e]      = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# ══════════════════════════════════════════════════════════════════════════
# آزمایش: آیا پنجره‌ی زمانی بلندتر (۶ یا ۹ ثانیه به‌جای ۳ ثانیه) اطلاعات
# کلاس بیشتری برای EEG آشکار می‌کند؟
# ══════════════════════════════════════════════════════════════════════════
#
# منطق ساخت پنجره‌ی بلندتر:
#   پنجره‌های ذخیره‌شده با گام ۱ ثانیه و طول ۳ ثانیه‌اند. پنجره‌ی index=i بازه‌ی
#   [i, i+3] ثانیه را پوشش می‌دهد و پنجره‌ی index=i+3 بازه‌ی [i+3, i+6] را —
#   یعنی این دو کاملاً پیوسته و بدون همپوشانی‌اند. با چسباندن N پنجره‌ی
#   غیرهم‌پوشان پشت‌سرهم (index i, i+3, i+6, ...) روی محور زمان، یک سیگنال
#   واقعی (نه تقریبی) به طول N×3 ثانیه بازسازی می‌شود.
#
# ⚠️ فرض مهم: این منطق فرض می‌کند اندیس‌های پشت‌سرهم در فایل ذخیره‌شده دقیقاً
#   با گام ۱ ثانیه به‌هم متصل‌اند (طبق طراحی windowing مقاله). بین trial ها
#   ممکن است این پیوستگی برقرار نباشد (rest period حذف شده). برای کاهش خطر
#   چسباندن دو تکه از trial متفاوت، فقط پنجره‌هایی با label یکسان به‌هم متصل
#   می‌شوند و پنجره‌های نامتجانس رد می‌شوند (تعدادشان گزارش می‌شود).

import scipy.signal as sig
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ── تنظیمات ──────────────────────────────────────────────────────────────
TEST_SESSION      = 1
FS                = 200
BASE_WIN_SEC      = 3          # طول هر پنجره‌ی خام ذخیره‌شده
STEP_SAMPLES      = 3          # گام اندیس برای غیرهم‌پوشان بودن (= BASE_WIN_SEC، چون گام زمانی خام ۱ثانیه است)
DURATIONS_TO_TEST = [1, 2, 3]  # ضریب N_STACK: 1→3s(baseline), 2→6s, 3→9s
STRIDE_BETWEEN_SAMPLES = 5     # هر چند اندیس یک نمونه‌ی جدید شروع کنیم (برای کنترل تعداد کل نمونه‌ها)
MAX_SAMPLES_PER_DURATION = 3000
N_FOLDS           = 5
RANDOM_STATE      = 42
WELCH_BATCH       = 150

BANDS = {
    'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13),
    'beta':  (13, 30), 'gamma': (30, 45),
}
# ──────────────────────────────────────────────────────────────────────────


def get_subject_blocks(dataset):
    """لیستی از (subject, start_idx, end_idx) به همان ترتیب ذخیره در X_eeg/y."""
    return [(f['subject'], f['start_idx'], f['end_idx']) for f in dataset.valid_files]


def iter_stacked_window_batches(dataset, n_stack, stride=STRIDE_BETWEEN_SAMPLES,
                                 max_samples=MAX_SAMPLES_PER_DURATION, batch_size=WELCH_BATCH):
    """
    FIX (MemoryError): به‌جای ساختن کل آرایه‌ی stacked (که برای duration=9s حجمش به
    چند گیگابایت می‌رسد)، این تابع یک generator است که هر بار فقط batch_size نمونه
    را می‌سازد و برمی‌گرداند. صدا‌زننده بلافاصله روی هر batch ویژگی‌های Welch را
    حساب می‌کند (که حجمش صدها برابر کوچک‌تر از سیگنال خام است) و batch خام را دور
    می‌ریزد -- پیک مصرف حافظه دیگر به تعداد کل نمونه‌ها وابسته نیست.
    """
    blocks = get_subject_blocks(dataset)
    batch_eeg, batch_labels, batch_subjects = [], [], []
    n_kept = 0
    n_skipped_label_mismatch = 0
    n_attempted = 0
    span_needed = (n_stack - 1) * STEP_SAMPLES + 1

    for subject, start, end in blocks:
        i = start
        while i + span_needed <= end and n_kept < max_samples:
            idxs = [i + k * STEP_SAMPLES for k in range(n_stack)]
            n_attempted += 1
            lbls = dataset.y[idxs]
            if not np.all(lbls == lbls[0]):
                n_skipped_label_mismatch += 1
                i += stride
                continue

            chunks = [dataset.X_eeg[idx] for idx in idxs]
            merged = np.concatenate(chunks, axis=-1)   # (16,16, n_stack*600)

            batch_eeg.append(merged)
            batch_labels.append(lbls[0])
            batch_subjects.append(subject)
            n_kept += 1
            i += stride

            if len(batch_eeg) >= batch_size:
                yield (np.stack(batch_eeg, axis=0), np.array(batch_labels), np.array(batch_subjects, dtype=object))
                batch_eeg, batch_labels, batch_subjects = [], [], []

        if n_kept >= max_samples:
            break

    if batch_eeg:  # آخرین batch ناقص
        yield (np.stack(batch_eeg, axis=0), np.array(batch_labels), np.array(batch_subjects, dtype=object))

    print(f"    duration={n_stack*BASE_WIN_SEC}s | attempted={n_attempted} | "
          f"skipped(label mismatch)={n_skipped_label_mismatch} | kept={n_kept}")


def compute_band_power_features(eeg_windows, fs=FS, batch_size=WELCH_BATCH):
    """همان منطق batched Welch از آزمایش قبلی -- طول محور زمان اینجا متغیر است."""
    n_windows, h, w, t_len = eeg_windows.shape
    n_channels = h * w
    x_full = eeg_windows.reshape(n_windows, n_channels, t_len)

    nperseg = min(256, t_len)   # با طول بلندتر، nperseg را هم کمی بزرگ‌تر می‌گیریم برای resolution بهتر
    noverlap = nperseg // 2

    broadband_chunks, band_chunks = [], []
    freqs = None
    for start in range(0, n_windows, batch_size):
        end = min(start + batch_size, n_windows)
        f, psd = sig.welch(x_full[start:end], fs=fs, nperseg=nperseg, noverlap=noverlap, axis=-1)
        if freqs is None:
            freqs = f

        bmask = (freqs >= 0.5) & (freqs <= 50)
        broadband_chunks.append(np.log10(psd[:, :, bmask].sum(axis=-1) + 1e-12))

        band_list = []
        for lo, hi in BANDS.values():
            mask = (freqs >= lo) & (freqs <= hi)
            band_list.append(np.log10(psd[:, :, mask].sum(axis=-1) + 1e-12))
        band_chunks.append(np.concatenate(band_list, axis=1))
        del psd

    return np.concatenate(broadband_chunks, axis=0), np.concatenate(band_chunks, axis=0)


def evaluate_features(broadband_feats, band_feats, labels, subjects):
    gkf = GroupKFold(n_splits=N_FOLDS)

    clf_bb = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
    clf_bp = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))

    scores_bb = cross_val_score(clf_bb, broadband_feats, labels, groups=subjects,
                                 cv=gkf, scoring='accuracy', n_jobs=-1)
    scores_bp = cross_val_score(clf_bp, band_feats, labels, groups=subjects,
                                 cv=gkf, scoring='accuracy', n_jobs=-1)
    return scores_bb, scores_bp


def run_duration_sweep():
    print(f"\nLoading raw EEG windows for session {TEST_SESSION} ...")
    unique_tag = f"DurationSweep_{TEST_SESSION}_{os.getpid()}"
    dataset = MMapHybridDataset(
        ALL_SUBJECTS, unique_tag,
        session_filter=[TEST_SESSION],
        min_windows_threshold=100
    )

    try:
        results = {}
        for n_stack in DURATIONS_TO_TEST:
            duration_sec = n_stack * BASE_WIN_SEC
            print(f"\n{'='*65}\nDuration = {duration_sec}s (stacking {n_stack} base windows)\n{'='*65}")

            # FIX (MemoryError): پردازش batch-به-batch -- هر batch از پنجره‌های خام
            # ساخته می‌شود، بلافاصله Welch رویش اجرا می‌شود (خروجی کوچک: broadband/band
            # power)، و batch خام دور ریخته می‌شود. فقط فیچرهای نهایی (خیلی کوچک‌تر از
            # سیگنال خام) در حافظه انباشته می‌شوند.
            bb_chunks, bp_chunks, lbl_chunks, subj_chunks = [], [], [], []
            for eeg_batch, lbl_batch, subj_batch in iter_stacked_window_batches(dataset, n_stack):
                bb, bp = compute_band_power_features(eeg_batch)
                bb_chunks.append(bb); bp_chunks.append(bp)
                lbl_chunks.append(lbl_batch); subj_chunks.append(subj_batch)
                del eeg_batch, bb, bp
            gc.collect()

            broadband_feats = np.concatenate(bb_chunks, axis=0)
            band_feats      = np.concatenate(bp_chunks, axis=0)
            labels          = np.concatenate(lbl_chunks, axis=0)
            subjects        = np.concatenate(subj_chunks, axis=0)
            print(f"  Class counts: {np.bincount(labels)} | subjects: {len(np.unique(subjects))}")

            scores_bb, scores_bp = evaluate_features(broadband_feats, band_feats, labels, subjects)
            results[duration_sec] = {
                'broadband_mean': scores_bb.mean() * 100, 'broadband_std': scores_bb.std() * 100,
                'bandpower_mean': scores_bp.mean() * 100, 'bandpower_std': scores_bp.std() * 100,
            }
            print(f"  Broadband: {scores_bb.mean()*100:.2f}% ± {scores_bb.std()*100:.2f}%")
            print(f"  Band-power: {scores_bp.mean()*100:.2f}% ± {scores_bp.std()*100:.2f}%")

            del broadband_feats, band_feats, labels, subjects
            gc.collect()

        print("\n" + "=" * 65)
        print("SUMMARY — Accuracy vs Window Duration (subject-grouped CV, linear classifier)")
        print("=" * 65)
        print(f"{'Duration':>10} | {'Broadband':>18} | {'Band-power':>18}")
        for dur, r in results.items():
            print(f"{dur:>8}s | {r['broadband_mean']:>6.2f}% ± {r['broadband_std']:>4.2f}%  | "
                  f"{r['bandpower_mean']:>6.2f}% ± {r['bandpower_std']:>4.2f}%")
        print(f"{'':>10} | {'chance = 33.33%':>18} |")

        print("\nتفسیر:")
        print("  - اگر دقت با افزایش طول پنجره (3s→6s→9s) به‌طور پیوسته و معنادار بالا رفت،")
        print("    یعنی مشکل EEG واقعاً از کوتاه بودن پنجره بوده -- می‌توانی طول پنجره‌ی")
        print("    TSFNet را افزایش دهی (نیازمند تغییر معماری input EEG branch).")
        print("  - اگر دقت تقریباً ثابت ماند (تغییرات داخل بازه‌ی std)، یعنی طول پنجره")
        print("    علت نبود؛ همراه با ۴ آزمایش قبلی، این EEG branch را به‌عنوان یک")
        print("    محدودیت شناخته‌شده‌ی دیتاست/تسک مستند کن و روی fusion تمرکز کن.")

        return results
    finally:
        dataset.cleanup()
        gc.collect()


if __name__ == "__main__":
    results = run_duration_sweep()


Loading raw EEG windows for session 1 ...
Scanning DurationSweep_1_2064 - 26 subjects (sessions [1])...
VP001 verified: 342 windows
VP002 verified: 342 windows
VP003 verified: 342 windows
VP004 verified: 231 windows
VP005 verified: 342 windows
VP006 verified: 342 windows
VP007 verified: 342 windows
VP008 verified: 342 windows
VP009 verified: 342 windows
VP010 verified: 342 windows
VP011 verified: 342 windows
VP012 verified: 342 windows
VP013 verified: 339 windows
VP014 verified: 336 windows
VP015 verified: 340 windows
VP016 verified: 342 windows
VP017 verified: 342 windows
VP018 verified: 342 windows
VP019 verified: 342 windows
VP020 verified: 342 windows
VP021 verified: 336 windows
VP022 verified: 342 windows
VP023 verified: 342 windows
VP024 verified: 342 windows
VP025 verified: 342 windows
VP026 verified: 342 windows
Streaming clean data to disk memory-map (DurationSweep_1_2064)...
--> DurationSweep_1_2064 Ready on Disk (CLEAN)! Total Windows: 8764 | Labels: [2927 2919 2918]


Dura